<img src = "../assets/bbai-banner.png" align="center" height = "25%">

In [ ]:
# Import local modules for use in this notebook
import os
import sys
module_paths = [
    os.path.abspath(os.path.join('..')),
]
for module_path in module_paths:
    if module_path not in sys.path:
        sys.path.append(module_path)

# Remaining Useful Life Demonstration #

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

### We have to identify the Columns of data that we will be using for training and evaluation.  
#### In some cases, it could make sense for this to be a subset of the whole.

In [ ]:
# Not all columns in the dataset is used. Select useful signals in the dataset
sensor_cols = ["s" + str(i) for i in set(range(1, 22))] + [
    "setting1",
    "setting2",
    "setting3",
]

### Prepare the training data, as in the Data Exploration Notebook

In [ ]:
# Specify the file path
filename = "FD001"
filepath = '../data/training/train_'+filename+'.txt'

# Set up Additional Parameters to be used in data exploration
# Maximum Remaining Useful Life value should be chosen for "no wear" on component.  This should also be chosen to leave enough time to order new component
max_RUL = 130

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

# Load the file into Pandas dataframe
train_data = pd.read_csv(filepath, sep=" ", header=None)
#train_data.drop(train_data.columns[[26, 27]], axis=1, inplace=True)
train_data.columns = column_names



### Instantiate the model

In [ ]:
from pdm.config import RULConfig
from pdm.rul_model import RULModel

cfg = RULConfig(sensor_columns=sensor_cols, save_file_name='../models/FD001.h5')
model = RULModel(**cfg.model_dump())


#### Batch Train the model and save the trained model to disk

In [ ]:
model.batch_train(train_data)

### Prepare the test data in the same way and predict using the trained model

In [ ]:
# Specify the file path
filepath = '../data/test/test_'+filename+'.txt'

# Set up Additional Parameters to be used in data exploration
# Maximum Remaining Useful Life value should be chosen for "no wear" on component.  This should also be chosen to leave enough time to order new component
max_RUL = 130

# Identify the Column Names for the data
column_names = ["id","cycle","setting1","setting2", "setting3",
        "s1","s2","s3","s4","s5","s6","s7","s8","s9","s10","s11","s12","s13","s14","s15","s16","s17","s18","s19","s20","s21",]

# Load the file into Pandas dataframe
test_data = pd.read_csv(filepath, sep=" ", header=None)
#test_data.drop(test_data.columns[[26, 27]], axis=1, inplace=True)
test_data.columns = column_names



test_data.head()

### Make the Predictions

In [ ]:
predictions = model.batch_predict(test_data)

print(predictions)

In [ ]:
#plot the prediction vs the ground truth
def plot_result(y_true, y_pred, title=None):

    """ A function to plot the predicted RUL vs the target RUL for all engines"""

    fig = plt.figure(figsize = (16, 12))
    plt.grid(True,lw=0.1)
    plt.rcParams['axes.linewidth'] = 0.4  # set the value globally
    plt.plot(y_true, 'r', linewidth=0.5,linestyle='--')
    plt.plot(y_pred, 'k', linewidth=0.5)
    plt.title(title,fontsize=8)
    plt.xlabel('Engine ID', fontsize=12)
    plt.xticks(fontsize=5)
    plt.ylabel('RUL', fontsize=12)
    plt.yticks(fontsize=5)
    plt.legend(['Actual RUL', 'Predicted RUL'],  loc='upper right', fontsize=5)
    plt.show()
    fig.savefig("../results/"  + title + '.tiff', dpi=90, format='tiff', bbox_inches='tight')
    return fig

### Plot the prediction results against the true results

In [ ]:
# Read the file and create a pandas Series
truth_file_path = "../data/RUL/RUL_FD001.txt"
truth_data = pd.read_csv(truth_file_path, header=None)
truth_data.columns=['trueRUL']

evaluation_df = pd.merge(predictions, truth_data, left_index=True, right_index=True)

evaluation_df = evaluation_df.sort_values('trueRUL', ascending=False).reset_index(drop=True)

plot_result(evaluation_df['trueRUL'], evaluation_df['predictions'], "Results")